# BhashaYantra GPU manuscript reader (live)
Runs the Modi manuscript model on Colab's free GPU and connects it to your website, so the Manuscript tab can read uploaded pages live.

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. In step 1 below, set `SITE` and `TOKEN` (the same token you set on Render as `MANUSCRIPT_GPU_TOKEN`)
3. `Runtime → Run all`, then keep this tab open while you demo. When you close it, the site shows "GPU reader offline".

In [ ]:
# 1. Your settings
SITE  = "https://bhashayantra.onrender.com"   # your website
TOKEN = "PASTE-THE-SAME-TOKEN-AS-ON-RENDER"   # keep this secret


In [ ]:
# 2. Install
!pip -q install -U transformers peft bitsandbytes accelerate fastapi uvicorn python-multipart
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
assert TOKEN and "PASTE" not in TOKEN, "Set TOKEN in step 1 (same as MANUSCRIPT_GPU_TOKEN on Render)"
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# 3. Load the model (4-bit, ~3-4 GB of GPU memory)
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel
BASE, ADAPTER = "Qwen/Qwen2.5-VL-3B-Instruct", "lgtk/qwen25vl-3b-modi-synth-lora"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(BASE, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER).eval()
processor = AutoProcessor.from_pretrained(BASE, max_pixels=512 * 28 * 28)
tok = processor.tokenizer
print("Model loaded.")


In [ ]:
# 4. Reading, with per-word confidence, image clean-up and loop protection
import cv2, numpy as np, torch.nn.functional as F, threading, time, io
from PIL import Image
MAX_NEW = 256
PROMPT = ("This image contains handwritten text in Modi script, a historical cursive "
          "script used to write the Marathi language. "
          "Transliterate the text in this image into Devanagari script. "
          "Output only the Devanagari text, with no explanation.")

def clean(img):
    g = cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2GRAY)
    g = cv2.fastNlMeansDenoising(g, h=10)
    g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)
    b = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15)
    return Image.fromarray(b).convert("RGB")

@torch.no_grad()
def read_once(img):
    msgs = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": PROMPT}]}]
    text_in = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text_in], images=[img], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=False, output_scores=True, return_dict_in_generate=True)
    gen = out.sequences[0, inputs["input_ids"].shape[1]:]
    probs = [F.softmax(s[0].float(), dim=-1)[t].item() for s, t in zip(out.scores, gen)]
    text = tok.decode(gen, skip_special_tokens=True).strip()
    conf = []
    for i, p in enumerate(probs):
        n = len(tok.decode(gen[: i + 1], skip_special_tokens=True).split())
        if n == 0: continue
        while len(conf) < n: conf.append(1.0)
        conf[n - 1] = min(conf[n - 1], p)
    words = text.split()
    return text, (conf + [1.0] * len(words))[: len(words)], len(gen) >= MAX_NEW - 1

def looks_looped(text: str, hit_limit: bool) -> bool:
    """A reading that ran to the token limit, or keeps repeating itself, is probably a loop."""
    words = text.split()
    if hit_limit:
        return True
    if len(words) >= 12 and len(set(words)) / len(words) < 0.5:
        return True
    grams = [" ".join(words[i:i + 3]) for i in range(len(words) - 2)]
    return any(grams.count(g) >= 3 for g in set(grams))

lock = threading.Lock()
def read_page(img):
    with lock:                                   # one page at a time on the GPU
        text, conf, hit = read_once(img)
        if not looks_looped(text, hit):
            return {"text": text, "word_conf": conf, "variant": "raw", "loop_detected": False}
        text2, conf2, hit2 = read_once(clean(img))  # page-27 finding: clean-up can break a loop
        use_clean = not looks_looped(text2, hit2) or len(text2) < len(text)
        return ({"text": text2, "word_conf": conf2, "variant": "cleaned", "loop_detected": True} if use_clean
                else {"text": text, "word_conf": conf, "variant": "raw", "loop_detected": True})
print("Ready.")


In [ ]:
# 5. Start the web server + secure public link, and register it with your site
import subprocess, re, requests, uvicorn
from fastapi import FastAPI, UploadFile, File, Header, HTTPException

api = FastAPI()

@api.get("/health")
def health(x_token: str = Header(default="")):
    if x_token != TOKEN: raise HTTPException(403)
    return {"ok": True}

@api.post("/read")
async def read(file: UploadFile = File(...), x_token: str = Header(default="")):
    if x_token != TOKEN: raise HTTPException(403)
    img = Image.open(io.BytesIO(await file.read())).convert("RGB")
    img.thumbnail((1600, 1600))
    t0 = time.time()
    out = read_page(img)
    out.update(model=ADAPTER, seconds=round(time.time() - t0, 1))
    print(f"read a page in {out['seconds']}s ({out['variant']}{', loop caught' if out['loop_detected'] else ''})")
    return out

server = uvicorn.Server(uvicorn.Config(api, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                          stderr=subprocess.PIPE, stdout=subprocess.DEVNULL, text=True)
PUBLIC = None
for line in tunnel.stderr:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m: PUBLIC = m.group(0); break
print("GPU link:", PUBLIC)
time.sleep(5)  # let the link go live

r = requests.post(f"{SITE}/manuscript/register", json={"url": PUBLIC}, headers={"X-Token": TOKEN}, timeout=120)
print("Registered with your site:", r.status_code, r.text)
print("\nOpen", SITE + "/app#scan", "- the upload box should now be live.")


In [ ]:
# 6. Keep running while you demo (stop this cell, or close the tab, to go offline)
while True:
    time.sleep(60)
    print(time.strftime("%H:%M"), "GPU reader running at", PUBLIC)
